# Gemma 3 Position-Geometry Localization

## v0.7.2 — fresh-seed six-condition follow-up

Gemma v0.7.1 did **not** replicate Qwen's positive canonical-ID position effect.
Instead, it produced a strong sign interaction:

- canonical IDs: `CE - CC < 0`
- fragmented IDs: `FE - FC > 0`

This notebook prospectively tests whether that Gemma-specific sign pattern reproduces on a
**disjoint semantic inventory** and then decomposes each total position effect into:

`source geometry → query/readout shift → filler positional spreading`

For canonical IDs:

`CC → CQ → CE`

For fragmented IDs:

`FC → FQ → FE`

### Frozen before this run

- model: `google/gemma-3-4b-it`
- 32 records
- target lengths: 4096 and 8192 canonical chat tokens
- filler languages: English and Turkish
- fragmentation strengths: EN `keep=1`, TR `keep=5`
- primary metric: gold-vs-best-distractor next-token logit margin
- scorer: one-shot, no-cache, custom sequential full/sliding visibility masks
- fresh semantic seeds: `6000..6031`
- bootstrap: 10,000 trial-cluster resamples, seed `20260828`
- no component direction is tuned or selected after seeing these results

### Primary sign-pattern replication rule

Before interpreting the decomposition, the v0.7.1 Gemma pattern must independently reproduce:

1. canonical CC accuracy ≥ 0.90;
2. canonical maximum predicted-digit share ≤ 0.40;
3. pooled `CE-CC < 0` and pooled `FE-FC > 0`;
4. the canonical total is negative in at least 3/4 language×length cells;
5. the fragmented total is positive in at least 3/4 cells;
6. every leave-one-trial-out mean preserves the corresponding sign.

The component decomposition is then reported with bootstrap CIs and prospective robust-sign
criteria; if the total sign pattern fails, component results remain descriptive rather than a
successful localization of the v0.7.1 phenomenon.

> Public reproduction copy: the prior v0.7.1 sign pattern and reproduction-essential model settings are frozen explicitly; internal calibration/debugging outputs are omitted.

## 0. GPU runtime

In [ ]:
import subprocess
subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    check=False,
)

## 1. Drive, provenance, and frozen v0.7.1 result

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, shutil
PROJECT_DIR=Path('/content/drive/MyDrive/gemma3_geometry_localization_v072')
PROJECT_DIR.mkdir(parents=True,exist_ok=True)
RESULTS_PATH=PROJECT_DIR/'v072_gemma_geometry_localization_4k8k.jsonl'
AUDIT_PATH=PROJECT_DIR/'v072_gemma_geometry_localization_audit.json'
ZIP_PATH=PROJECT_DIR/'fragburden0.3.0.zip'
REPO_DIR=PROJECT_DIR/'fragburden_src'/'fragburden'
MODEL_NAME='google/gemma-3-4b-it'
TRANSFORMERS_VERSION='5.15.1'
NOTEBOOK_VERSION='v0.7.2-public-repro'
TASK_VERSION='symbolic_assoc_r1_gemma_v072'
SCORER_VERSION='gemma3_oneshot_nocache_margin_v072'
POSITION_INTERVENTION='query_shift_vs_filler_spread_gemma_v1'
N_RECORDS=32
TARGET_LENGTHS=[4096,8192]
FILLER_LANGUAGES=['en','tr']
SEMANTIC_SEEDS=list(range(6000,6032))
POSITION_RANK_SET=[0,2,5,7]
BOOTSTRAP_REPS=10000
BOOTSTRAP_SEED=20260828
engineering={'model': 'google/gemma-3-4b-it', 'model_class': 'Gemma3ForConditionalGeneration', 'transformers_version_actual': '5.15.1', 'frozen_n_records': 32, 'frag_keep_by_language': {'en': 1, 'tr': 5}, 'target_lengths': [4096, 8192], 'reserved_replication_seeds': [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031], 'confirmatory_scorer': 'one_shot_no_cache_custom_masks', 'precision_mode': {'weights': 'NF4', 'bnb_4bit_compute_dtype': 'bfloat16', 'nonquantized_dtype': 'bfloat16'}, 'layer_type_counts': {'full_attention': 5, 'sliding_attention': 29}, 'digit_ids': {'1': 236770, '2': 236778, '3': 236800, '4': 236812}, 'chat_prefix_tokens': 4, 'chat_suffix_tokens': 5, 'fragmentation_capability': {'dial_type': 'merge_rank'}}
frozen=engineering  # compatibility with model-loader checks
previous={'observed_rows': 512, 'cross_model_position_replication': False, 'frozen_frag_keep_by_language': {'en': 1, 'tr': 5}, 'pooled_effects_cluster_bootstrap': {'position_effect_canonical_ids': {'mean': -11.11279296875, 'ci_lo': -12.30323486328125, 'ci_hi': -9.7599609375}, 'position_effect_fragmented_ids': {'mean': 2.17822265625, 'ci_lo': 1.66990966796875, 'ci_hi': 2.65382080078125}}}
prior_effects=previous['pooled_effects_cluster_bootstrap']
prior_can=prior_effects['position_effect_canonical_ids']
prior_frag=prior_effects['position_effect_fragmented_ids']
FRAG_KEEP_BY_LANGUAGE={'en':1,'tr':5}
print('FROZEN PAPER SETTINGS LOADED; fresh seeds =', SEMANTIC_SEEDS)


## 2. Install frozen software stack and authenticate

In [ ]:
import subprocess, sys, importlib, os
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN secret detected.')
else:
    print('WARNING: set Colab secret HF_TOKEN before loading gated Gemma weights.')

if not ZIP_PATH.exists():
    repo_zip = Path('/content/token-distance-positional-geometry/code/fragburden0.3.0.zip')
    if repo_zip.exists(): shutil.copy2(repo_zip, ZIP_PATH)
    else:
        from google.colab import files
        uploaded=files.upload(); zs=[n for n in uploaded if n.endswith('.zip')]
        if not zs: raise RuntimeError('Upload code/fragburden0.3.0.zip from the repository.')
        shutil.copy2(Path(zs[0]), ZIP_PATH)
extract_root=PROJECT_DIR/'fragburden_src'
if extract_root.exists(): shutil.rmtree(extract_root)
extract_root.mkdir(parents=True,exist_ok=True)
shutil.unpack_archive(str(ZIP_PATH),str(extract_root))
# Locate pyproject in extracted archive.
candidates=list(extract_root.rglob('pyproject.toml'))
if not candidates: raise RuntimeError('pyproject.toml not found in fragburden archive.')
REPO_DIR=candidates[0].parent
subprocess.check_call([sys.executable,'-m','pip','install','-e',f'{REPO_DIR}[gpu,analysis]'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-U',f'transformers=={TRANSFORMERS_VERSION}','accelerate','bitsandbytes','huggingface_hub'])
print('Installation complete.')


## 3. Load Gemma 3 in the validated BF16/NF4 configuration

In [ ]:
import hashlib, random, math, inspect, sys, gc, time, importlib
from dataclasses import dataclass

import numpy as np
import pandas as pd

# Defensive path repair for notebook runtimes.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

importlib.invalidate_caches()

import torch
import torch.nn.functional as F
import transformers

import fragburden
from fragburden.dial import FragmentationDial, verify_monotone
from fragburden.testing import sample_texts

from transformers import (
    AutoTokenizer,
    Gemma3ForConditionalGeneration,
    BitsAndBytesConfig,
    DynamicCache,
)

assert transformers.__version__ == TRANSFORMERS_VERSION
assert torch.cuda.is_available()

gpu_name = torch.cuda.get_device_name(0)
cuda_capability = torch.cuda.get_device_capability()
bf16_supported = bool(torch.cuda.is_bf16_supported())

print('fragburden:', fragburden.__file__)
print('transformers:', transformers.__version__)
print('torch:', torch.__version__)
print('GPU:', gpu_name)
print('CUDA capability:', cuda_capability)
print('Native BF16 supported:', bf16_supported)

if not bf16_supported:
    raise RuntimeError(
        'This Gemma 3 replication notebook requires a runtime for which '
        'PyTorch reports BF16 support. The current runtime does not. '
        'Do not silently fall back to FP16.'
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    token=hf_token,
)

# Gemma 3 4B is distributed as a multimodal checkpoint. Load the matching
# wrapper so checkpoint names are mapped correctly, even though our experiment
# supplies text only.
#
# Keep the large language-model weights in NF4, but use BF16 for 4-bit compute
# and all non-quantized modules. This matches the model's intended activation
# precision and avoids the FP16 NaN path observed earlier.
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

load_kwargs = dict(
    quantization_config=bnb,
    device_map='auto',
    attn_implementation='sdpa',
    token=hf_token,
)

try:
    model = Gemma3ForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        dtype=torch.bfloat16,
        **load_kwargs,
    )
except TypeError:
    model = Gemma3ForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        **load_kwargs,
    )

model.eval()

if hasattr(model.config, 'use_cache'):
    model.config.use_cache = True

device = next(model.parameters()).device
text_config = model.config.text_config

sliding_window = int(text_config.sliding_window)

layer_types = getattr(text_config, 'layer_types', None)

if layer_types is None:
    pattern = int(getattr(text_config, 'sliding_window_pattern', 6))
    layer_types = [
        (
            'full_attention'
            if ((i + 1) % pattern == 0)
            else 'sliding_attention'
        )
        for i in range(int(text_config.num_hidden_layers))
    ]

layer_types = list(layer_types)

if not {'full_attention', 'sliding_attention'}.issubset(set(layer_types)):
    raise RuntimeError(
        f'Unexpected Gemma layer types: {set(layer_types)}'
    )

# Full multimodal wrapper must expose position_ids because that is the public
# entry point used by our scorer.
if 'position_ids' not in inspect.signature(model.forward).parameters:
    raise RuntimeError(
        'Gemma3ForConditionalGeneration forward does not expose position_ids.'
    )

# Check that we actually loaded the matching architecture rather than silently
# constructing an uninitialized text model.
loaded_param_names = [name for name, _ in model.named_parameters()]
if not any('language_model' in name for name in loaded_param_names):
    raise RuntimeError(
        'Loaded Gemma wrapper does not expose language-model parameters.'
    )


print('Loaded:', MODEL_NAME)
print('model class:', type(model).__name__)
print('model dtype:', model.dtype)
print('text max positions:', text_config.max_position_embeddings)
print('sliding window:', sliding_window)
layer_type_counts = {
    k: int(layer_types.count(k))
    for k in sorted(set(layer_types))
}
print('layer type counts:', layer_type_counts)
print('precision mode:', engineering['precision_mode'])
if layer_type_counts != {
    str(k): int(v)
    for k, v in engineering['layer_type_counts'].items()
}:
    raise RuntimeError(
        f'Gemma layer-type counts drifted: {layer_type_counts} '
        f'!= {frozen["layer_type_counts"]}'
    )

print('MODEL LOAD GATE PASSED.')

if type(model).__name__ != engineering['model_class']:
    raise RuntimeError(
        f'Model class drift: {type(model).__name__} != {frozen["model_class"]}'
    )

if transformers.__version__ != engineering['transformers_version_actual']:
    raise RuntimeError(
        'Transformers version drift from engineering audit.'
    )

print('MODEL LOAD / PRECISION GATE PASSED.')

total_vram_gb=torch.cuda.get_device_properties(0).total_memory/(1024**3)
if total_vram_gb < 20: raise RuntimeError('Use a >=20 GiB GPU (L4/A100) for exact 8K fragmented reproduction.')
print('GPU memory gate passed:', total_vram_gb)


## 4. Frozen fragmentation dial

In [ ]:
dial = FragmentationDial.from_transformers(
    tokenizer,
    name=MODEL_NAME,
)

cap = dial.capability()

if not cap.get(
    'usable_for_arm_a'
):
    raise RuntimeError(
        'No usable Gemma fragmentation dial.'
    )

if cap.get(
    'dial_type'
) != engineering[
    'fragmentation_capability'
].get(
    'dial_type'
):
    raise RuntimeError(
        'Fragmentation dial type drifted.'
    )

d100 = dial.at(100)

DIALS = {
    language: dial.at(
        FRAG_KEEP_BY_LANGUAGE[
            language
        ]
    )
    for language in FILLER_LANGUAGES
}

for language, d in DIALS.items():
    print(
        language,
        {
            'keep': FRAG_KEEP_BY_LANGUAGE[
                language
            ],
            'dial_type': d.dial_type,
            'fingerprint': d.fingerprint,
        },
    )

print('FROZEN FRAGMENTATION DIALS READY.')

## 5. Official Gemma chat shell and answer-token gate

In [ ]:
DIGITS = list('1234')

def _normalize_input_ids(obj):
    if hasattr(obj, 'keys') and 'input_ids' in obj:
        obj = obj['input_ids']

    if torch.is_tensor(obj):
        obj = obj.detach().cpu().tolist()

    if (
        isinstance(obj, (list, tuple))
        and len(obj) == 1
        and isinstance(obj[0], (list, tuple))
    ):
        obj = obj[0]

    return [int(x) for x in obj]

def apply_chat_ids(user_content):
    kwargs = dict(
        tokenize=True,
        add_generation_prompt=True,
    )

    try:
        out = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': user_content}],
            return_dict=True,
            **kwargs,
        )
    except TypeError:
        out = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': user_content}],
            **kwargs,
        )

    return _normalize_input_ids(out)

def render_chat(user_content):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_content}],
        tokenize=False,
        add_generation_prompt=True,
    )

def assistant_continuation_ids(user_content, answer_text):
    prompt = apply_chat_ids(user_content)

    full = tokenizer.apply_chat_template(
        [
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': answer_text},
        ],
        tokenize=True,
        add_generation_prompt=False,
    )
    full = _normalize_input_ids(full)

    if full[:len(prompt)] != prompt:
        raise RuntimeError(
            'Assistant continuation does not begin with generation-prompt IDs.'
        )

    continuation = full[len(prompt):]
    if not continuation:
        raise RuntimeError('Empty assistant continuation.')

    return continuation

PROBE_USER = (
    'Task: look up the value for the queried key.\n'
    'EVIDENCE:\n'
    'K10000=1\nK10001=2\nK10002=3\nK10003=4\n'
    'QUERY KEY: K10000\n'
    'Answer with one digit only:'
)

DIGIT_IDS = {}
DIGIT_CONTINUATIONS = {}

for d in DIGITS:
    continuation = assistant_continuation_ids(PROBE_USER, d)
    DIGIT_CONTINUATIONS[d] = continuation
    DIGIT_IDS[d] = int(continuation[0])

if len(set(DIGIT_IDS.values())) != 4:
    raise RuntimeError(f'Digit first-token IDs are not distinct: {DIGIT_IDS}')

SHELL_MARKER = 'UNIQUE_USER_CONTENT_MARKER_GEMMA_V070_A93F27'
rendered_marker = render_chat(SHELL_MARKER)

if rendered_marker.count(SHELL_MARKER) != 1:
    raise RuntimeError('Could not isolate Gemma user-content shell.')

marker_idx = rendered_marker.index(SHELL_MARKER)
CHAT_PREFIX_TEXT = rendered_marker[:marker_idx]
CHAT_SUFFIX_TEXT = rendered_marker[marker_idx + len(SHELL_MARKER):]

CHAT_PREFIX_IDS = tokenizer(
    CHAT_PREFIX_TEXT,
    add_special_tokens=False,
)['input_ids']

CHAT_SUFFIX_IDS = tokenizer(
    CHAT_SUFFIX_TEXT,
    add_special_tokens=False,
)['input_ids']

def canonical_user_encoding(user_content):
    enc = tokenizer(
        user_content,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    return {
        'ids': [int(x) for x in enc['input_ids']],
        'offsets': [
            (int(a), int(b))
            for a, b in enc['offset_mapping']
        ],
    }

def verify_chat_shell(user_content):
    enc = canonical_user_encoding(user_content)
    reconstructed = CHAT_PREFIX_IDS + enc['ids'] + CHAT_SUFFIX_IDS
    official = apply_chat_ids(user_content)

    if reconstructed != official:
        raise RuntimeError(
            'Gemma chat-shell reconstruction failed. '
            'Do not continue to causal intervention.'
        )

    return {
        'official_ids': official,
        'canonical_user_ids': enc['ids'],
    }

verify_chat_shell(PROBE_USER)


print('Digit IDs:', DIGIT_IDS)
print('Digit continuation decodes:', {
    d: tokenizer.decode(
        ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )
    for d, ids in DIGIT_CONTINUATIONS.items()
})
print('Prefix tokens:', len(CHAT_PREFIX_IDS))
print('Suffix tokens:', len(CHAT_SUFFIX_IDS))
print('CHAT INTERFACE GATE PASSED.')

if DIGIT_IDS != {
    str(k): int(v)
    for k, v in engineering['digit_ids'].items()
}:
    raise RuntimeError(
        f'Digit-token IDs drifted from engineering audit: {DIGIT_IDS}'
    )

if len(CHAT_PREFIX_IDS) != int(engineering['chat_prefix_tokens']):
    raise RuntimeError('Chat-prefix token count drifted.')

if len(CHAT_SUFFIX_IDS) != int(engineering['chat_suffix_tokens']):
    raise RuntimeError('Chat-suffix token count drifted.')

print('FROZEN CHAT INTERFACE GATE PASSED.')

if DIGIT_IDS != {
    str(k): int(v)
    for k, v in engineering[
        'digit_ids'
    ].items()
}:
    raise RuntimeError(
        f'Digit IDs drifted: {DIGIT_IDS}'
    )

if len(
    CHAT_PREFIX_IDS
) != int(
    engineering[
        'chat_prefix_tokens'
    ]
):
    raise RuntimeError(
        'Chat-prefix token count drifted.'
    )

if len(
    CHAT_SUFFIX_IDS
) != int(
    engineering[
        'chat_suffix_tokens'
    ]
):
    raise RuntimeError(
        'Chat-suffix token count drifted.'
    )

print(
    'FROZEN CHAT INTERFACE GATE PASSED.'
)


## 6. Frozen associative-retrieval generator

In [ ]:
@dataclass
class CharSpan:
    start: int
    end: int
    label: str

@dataclass
class AssocInstance:
    filler_language: str
    semantic_seed: int
    trial_index: int
    target_chat_tokens: int
    user_content: str
    spans: list
    target_key: str
    gold_digit: str
    target_record_position: int
    evidence_lines: list
    filler_lines: list
    query_lines: list

def stable_int(*parts):
    blob = '|'.join(map(str, parts)).encode('utf-8')
    return int.from_bytes(
        hashlib.sha256(blob).digest()[:8],
        'big',
    )

def stable_rng(*parts):
    return random.Random(stable_int(*parts))

# Keep the v0.6.1 semantic generator tags so seeds 2000..2031
# denote the same item inventory across Qwen and Gemma.
def make_key(semantic_seed, slot):
    value = (
        stable_int(
            'key_v061',
            semantic_seed,
            slot,
            N_RECORDS,
        )
        % 90000
        + 10000
    )
    return f'K{value}'

def gold_for_trial(trial_index):
    return DIGITS[trial_index % 4]

def position_rank_for_trial(trial_index):
    gold_index = trial_index % 4
    replicate = trial_index // 4

    return POSITION_RANK_SET[
        (replicate + gold_index)
        % len(POSITION_RANK_SET)
    ]

def target_position_for_trial(trial_index):
    rank = position_rank_for_trial(trial_index)

    return int(
        round(
            rank
            * (N_RECORDS - 1)
            / 7
        )
    )

FILLER_BANK = {
    'en': {
        'subjects': [
            'The archivist', 'The technician', 'The editor',
            'The visitor', 'The clerk', 'The gardener',
        ],
        'verbs': [
            'reviewed', 'checked', 'organized',
            'recorded', 'sorted', 'noted',
        ],
        'objects': [
            'a routine report', 'the daily schedule', 'a short memo',
            'the supply list', 'a maintenance note', 'the meeting agenda',
        ],
        'settings': [
            'near the window', 'in the quiet office', 'before lunch',
            'during the afternoon', 'at the front desk', 'after the meeting',
        ],
    },
    'tr': {
        'subjects': [
            'Arşiv görevlisi', 'Teknisyen', 'Editör',
            'Ziyaretçi', 'Görevli', 'Bahçıvan',
        ],
        'verbs': [
            'inceledi', 'kontrol etti', 'düzenledi',
            'kaydetti', 'sıraladı', 'not etti',
        ],
        'objects': [
            'sıradan bir raporu', 'günlük programı', 'kısa bir notu',
            'malzeme listesini', 'bakım notunu', 'toplantı gündemini',
        ],
        'settings': [
            'pencerenin yanında', 'sessiz ofiste', 'öğle yemeğinden önce',
            'öğleden sonra', 'ön masada', 'toplantıdan sonra',
        ],
    },
}

def filler_sentence(language, idx, semantic_seed):
    bank = FILLER_BANK[language]
    x = idx + 7919 * semantic_seed

    subject = bank['subjects'][x % len(bank['subjects'])]
    x //= len(bank['subjects'])

    verb = bank['verbs'][x % len(bank['verbs'])]
    x //= len(bank['verbs'])

    obj = bank['objects'][x % len(bank['objects'])]
    x //= len(bank['objects'])

    setting = bank['settings'][x % len(bank['settings'])]

    if language == 'en':
        return f'{subject} {verb} {obj} {setting}.'

    return f'{subject} {setting} {obj} {verb}.'

def make_records(semantic_seed, trial_index):
    gold = gold_for_trial(trial_index)
    values = DIGITS * (N_RECORDS // 4)

    gold_idx = values.index(gold)
    values[0], values[gold_idx] = values[gold_idx], values[0]

    records = [
        (make_key(semantic_seed, slot), values[slot])
        for slot in range(N_RECORDS)
    ]

    target = records[0]
    others = records[1:]

    stable_rng(
        'records_v061',
        semantic_seed,
        trial_index,
    ).shuffle(others)

    pos = target_position_for_trial(trial_index)
    ordered = others[:]
    ordered.insert(pos, target)

    return ordered, target[0], gold, pos

def base_evidence_query(semantic_seed, trial_index):
    records, target_key, gold, pos = make_records(
        semantic_seed,
        trial_index,
    )

    evidence = [
        'Task: look up the value for the queried key.',
        'Each evidence line has the form KEY=VALUE.',
        'Return only the VALUE associated with the queried KEY.',
        'Do not explain or repeat the key.',
        'EVIDENCE:',
    ]

    evidence += [
        f'{key}={value}'
        for key, value in records
    ]

    query = [
        f'QUERY KEY: {target_key}',
        'Answer with one digit only:',
    ]

    return evidence, query, target_key, gold, pos

def compose_user_content(evidence, filler, query):
    evidence_text = '\n'.join(evidence)
    filler_text = '\n'.join(filler)
    query_text = '\n'.join(query)

    user_content = (
        evidence_text
        + '\n'
        + filler_text
        + '\n'
        + query_text
    )

    filler_start = len(evidence_text) + 1
    query_start = filler_start + len(filler_text) + 1

    spans = [
        CharSpan(0, filler_start, 'evidence'),
        CharSpan(filler_start, query_start, 'filler'),
        CharSpan(query_start, len(user_content), 'query'),
    ]

    return user_content, spans

def build_instance(
    filler_language,
    semantic_seed,
    trial_index,
    target_chat_tokens,
):
    evidence, query, target_key, gold, pos = base_evidence_query(
        semantic_seed,
        trial_index,
    )

    filler = []

    while True:
        user_content, spans = compose_user_content(
            evidence,
            filler,
            query,
        )

        if len(apply_chat_ids(user_content)) >= target_chat_tokens:
            break

        filler.append(
            filler_sentence(
                filler_language,
                len(filler),
                semantic_seed,
            )
        )

        if len(filler) > 5000:
            raise RuntimeError('Filler loop failed.')

    verify_chat_shell(user_content)

    return AssocInstance(
        filler_language=filler_language,
        semantic_seed=semantic_seed,
        trial_index=trial_index,
        target_chat_tokens=target_chat_tokens,
        user_content=user_content,
        spans=spans,
        target_key=target_key,
        gold_digit=gold,
        target_record_position=pos,
        evidence_lines=evidence,
        filler_lines=filler,
        query_lines=query,
    )

print('Gemma associative generator ready.')

## 7. Gemma-safe filler fragmentation

In [ ]:
def overlap_len(a, b, span):
    return max(
        0,
        min(b, span.end) - max(a, span.start),
    )

def token_region(a, b, spans):
    overlaps = {
        s.label: overlap_len(a, b, s)
        for s in spans
    }

    if max(overlaps.values()) <= 0:
        raise RuntimeError(f'Token offset [{a},{b}) has no semantic region.')

    return max(overlaps, key=overlaps.get)

def fully_inside_filler(a, b, spans):
    filler = next(s for s in spans if s.label == 'filler')
    return filler.start <= a and b <= filler.end

def build_user_encoding(inst, dfrag):
    enc = canonical_user_encoding(inst.user_content)

    canonical_ids = enc['ids']
    offsets = enc['offsets']

    if len(canonical_ids) != len(offsets):
        raise RuntimeError('IDs/offsets length mismatch.')

    canonical_labels = []
    fragmented_ids = []
    fragmented_labels = []
    fragmented_source_positions = []

    boundary_kept = 0

    for canonical_idx, (cid, (a, b)) in enumerate(
        zip(canonical_ids, offsets)
    ):
        label = token_region(a, b, inst.spans)
        canonical_labels.append(label)

        piece = tokenizer.convert_ids_to_tokens(int(cid))

        if label == 'filler' and fully_inside_filler(a, b, inst.spans):
            f_pieces = dfrag.encode_chunk_pieces(piece)

            if ''.join(f_pieces) != piece:
                raise RuntimeError(
                    f'Fragmented pieces do not reconstruct canonical piece {piece!r}.'
                )

            f_ids = d100.pieces_to_ids(f_pieces)
            m = len(f_ids)

            if m < 1:
                raise RuntimeError('Empty fragmentation.')

            src = [
                canonical_idx + rank / m
                for rank in range(m)
            ]
        else:
            if label == 'filler' and not fully_inside_filler(a, b, inst.spans):
                boundary_kept += 1

            f_ids = [int(cid)]
            src = [float(canonical_idx)]

        fragmented_ids.extend(f_ids)
        fragmented_labels.extend([label] * len(f_ids))
        fragmented_source_positions.extend(src)

    canonical_decoded = tokenizer.decode(
        canonical_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    fragmented_decoded = tokenizer.decode(
        fragmented_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    if fragmented_decoded != canonical_decoded:
        raise RuntimeError(
            'Gemma fragmented user IDs do not decode identically to canonical IDs.'
        )

    pos = np.asarray(
        fragmented_source_positions,
        dtype=np.float32,
    )

    if len(pos) > 1 and not np.all(np.diff(pos) > 0):
        raise RuntimeError('Fragmented source positions are not strictly increasing.')

    return {
        'canonical_ids': canonical_ids,
        'fragmented_ids': fragmented_ids,
        'canonical_labels': canonical_labels,
        'fragmented_labels': fragmented_labels,
        'fragmented_source_positions': pos,
        'boundary_kept_tokens': boundary_kept,
        'canonical_decoded': canonical_decoded,
    }

print('Gemma token-level fragmentation helper ready.')

## 8. Validated one-shot/no-cache scorer

In [ ]:
def _gemma3_sequential_masks(start, end, device):
    """
    Build Gemma 3's two attention masks from SEQUENTIAL TOKEN ORDER,
    independently of experimental position_ids.

    IMPORTANT:
    Transformers' Gemma 3 attention path expects 4-D ADDITIVE masks:
      * visible entries = 0
      * masked entries  = a large negative finite value

    A boolean dict mask is not equivalent here. Passing True/False directly
    caused NaNs in SDPA in v0.7.0.1 because Gemma bypasses its internal mask
    construction whenever attention_mask is already a dict.
    """
    q = torch.arange(
        start,
        end,
        device=device,
    )[:, None]

    k = torch.arange(
        0,
        end,
        device=device,
    )[None, :]

    full_visible = (k <= q)

    sliding_visible = (
        full_visible
        & ((q - k) < sliding_window)
    )

    # Gemma's text embeddings stay floating-point even when linear layers
    # are 4-bit quantized. SDPA expects a float mask compatible with Q/K/V.
    mask_dtype = torch.bfloat16

    min_value = torch.finfo(mask_dtype).min

    full = torch.full(
        (end - start, end),
        min_value,
        dtype=mask_dtype,
        device=device,
    )
    full.masked_fill_(full_visible, 0.0)

    local = torch.full(
        (end - start, end),
        min_value,
        dtype=mask_dtype,
        device=device,
    )
    local.masked_fill_(sliding_visible, 0.0)

    full = full[None, None, :, :]
    local = local[None, None, :, :]

    # Hard engineering checks: no NaNs, correct convention.
    if not torch.isfinite(full).all():
        raise RuntimeError('Non-finite values in Gemma full-attention mask.')

    if not torch.isfinite(local).all():
        raise RuntimeError('Non-finite values in Gemma sliding-attention mask.')

    if float(full[0, 0, -1, -1].item()) != 0.0:
        raise RuntimeError('Gemma full mask diagonal must be visible (0).')

    if float(local[0, 0, -1, -1].item()) != 0.0:
        raise RuntimeError('Gemma local mask diagonal must be visible (0).')

    return {
        'full_attention': full,
        'sliding_attention': local,
    }
def _candidate_view(logits, gold_digit):
    logits = logits.float()

    candidate_ids = torch.tensor(
        [DIGIT_IDS[d] for d in DIGITS],
        dtype=torch.long,
        device=logits.device,
    )

    candidate_logits = logits.index_select(
        0,
        candidate_ids,
    )

    if not torch.isfinite(candidate_logits).all():
        bad = candidate_logits.detach().float().cpu().tolist()
        raise RuntimeError(
            'Non-finite Gemma candidate logits. '
            f'Candidate logits={bad}. '
            'This indicates an invalid attention/position execution path, '
            'not a meaningful model prediction.'
        )

    candidate_logp = F.log_softmax(
        candidate_logits,
        dim=-1,
    )
    candidate_prob = candidate_logp.exp()

    gold_idx = DIGITS.index(gold_digit)

    distractor_indices = [
        i for i in range(len(DIGITS))
        if i != gold_idx
    ]

    best_distractor_idx = max(
        distractor_indices,
        key=lambda i: float(candidate_logits[i].item()),
    )

    pred_idx = int(torch.argmax(candidate_logits).item())
    pred = DIGITS[pred_idx]

    full_argmax_id = int(torch.argmax(logits).item())
    full_argmax_digit = next(
        (
            d for d, tok_id in DIGIT_IDS.items()
            if tok_id == full_argmax_id
        ),
        None,
    )

    return {
        'predicted_digit': pred,
        'correct': int(pred == gold_digit),
        'gold_logit_margin': float(
            (
                candidate_logits[gold_idx]
                - candidate_logits[best_distractor_idx]
            ).item()
        ),
        'gold_logit': float(candidate_logits[gold_idx].item()),
        'best_distractor_digit': DIGITS[best_distractor_idx],
        'best_distractor_logit': float(
            candidate_logits[best_distractor_idx].item()
        ),
        'forced_choice_nll': float(
            -candidate_logp[gold_idx].item()
        ),
        'gold_candidate_probability': float(
            candidate_prob[gold_idx].item()
        ),
        'full_argmax_id': full_argmax_id,
        'full_argmax_text': tokenizer.decode(
            [full_argmax_id],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ),
        'full_argmax_digit': full_argmax_digit,
        'full_argmax_is_gold': int(
            full_argmax_digit == gold_digit
        ),
        'candidate_logits': {
            d: float(candidate_logits[i].item())
            for i, d in enumerate(DIGITS)
        },
        'candidate_probabilities': {
            d: float(candidate_prob[i].item())
            for i, d in enumerate(DIGITS)
        },
    }

def score_one_shot_custom(
    input_ids,
    position_ids,
    gold_digit,
):
    """
    Confirmatory Gemma scorer.

    No KV cache and no chunking are used. Visibility is determined only by
    sequential token order via the custom full/sliding additive masks;
    experimental position_ids affect RoPE coordinates independently.
    """
    if len(input_ids) != len(position_ids):
        raise ValueError('IDs/positions length mismatch.')

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    ids = torch.tensor(
        input_ids,
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    pos = torch.tensor(
        position_ids,
        dtype=torch.float32,
        device=device,
    ).unsqueeze(0)

    n = ids.shape[1]

    masks = _gemma3_sequential_masks(
        0,
        n,
        device,
    )

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.inference_mode():
        out = model(
            input_ids=ids,
            attention_mask=masks,
            position_ids=pos,
            use_cache=False,
            logits_to_keep=1,
            return_dict=True,
        )

    logits = out.logits[0, -1, :]
    scored = _candidate_view(
        logits,
        gold_digit,
    )

    torch.cuda.synchronize()

    scored['seconds'] = time.perf_counter() - t0
    scored['peak_cuda_gb'] = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )
    scored['scorer_version'] = (
        SCORER_VERSION
        + '_oneshot_nocache'
    )

    del ids, pos, masks, out, logits
    gc.collect()
    torch.cuda.empty_cache()

    return scored

def flatten_score(scored):
    out = {
        k: v
        for k, v in scored.items()
        if k not in {
            'candidate_logits',
            'candidate_probabilities',
        }
    }

    for d, v in scored['candidate_logits'].items():
        out[f'logit_{d}'] = v

    for d, v in scored['candidate_probabilities'].items():
        out[f'prob_{d}'] = v

    return out

print('CONFIRMATORY ONE-SHOT SCORER READY.')

def flatten_score(scored):
    out = {
        k: v
        for k, v in scored.items()
        if k not in {
            'candidate_logits',
            'candidate_probabilities',
        }
    }

    for d, v in scored['candidate_logits'].items():
        out[f'logit_{d}'] = v

    for d, v in scored['candidate_probabilities'].items():
        out[f'prob_{d}'] = v

    return out

print('CONFIRMATORY ONE-SHOT SCORER READY.')

## 9. Six position maps

The decomposition is exact by construction:

- canonical: `CE - CC = (CQ - CC) + (CE - CQ)`
- fragmented: `FE - FC = (FQ - FC) + (FE - FQ)`

`CQ` and `CE` have the same canonical token IDs and the same query/readout positions; only
filler positions differ. `FQ` and `FE` satisfy the analogous fragmented-ID invariant.

In [ ]:
def hash_ids(ids):
    return hashlib.sha256(
        np.asarray(ids, dtype=np.int64).tobytes()
    ).hexdigest()[:24]

def hash_positions(pos):
    return hashlib.sha256(
        np.asarray(pos, dtype=np.float32).tobytes()
    ).hexdigest()[:24]

def full_gap_diagnostics(
    positions,
    prefix_len,
    labels,
):
    labels = np.asarray(labels)

    evidence_idx = np.where(labels == 'evidence')[0]
    filler_idx = np.where(labels == 'filler')[0]
    query_idx = np.where(labels == 'query')[0]

    if not (
        len(evidence_idx)
        and len(filler_idx)
        and len(query_idx)
    ):
        raise RuntimeError(
            'Missing evidence/filler/query region.'
        )

    evidence_last = float(
        positions[prefix_len + evidence_idx[-1]]
    )
    filler_first = float(
        positions[prefix_len + filler_idx[0]]
    )
    filler_last = float(
        positions[prefix_len + filler_idx[-1]]
    )
    query_first = float(
        positions[prefix_len + query_idx[0]]
    )
    query_last = float(
        positions[prefix_len + query_idx[-1]]
    )
    readout = float(positions[-1])

    return {
        'evidence_query': query_first - evidence_last,
        'evidence_readout': readout - evidence_last,
        'filler_span': filler_last - filler_first,
        'filler_query_gap': query_first - filler_last,
        'query_readout': readout - query_last,
    }

def assemble_geometry(inst):
    shell = verify_chat_shell(inst.user_content)

    dfrag = DIALS[inst.filler_language]
    enc = build_user_encoding(inst, dfrag)

    c_user = enc['canonical_ids']
    f_user = enc['fragmented_ids']

    c_labels = np.asarray(enc['canonical_labels'])
    f_labels = np.asarray(enc['fragmented_labels'])

    prefix_len = len(CHAT_PREFIX_IDS)
    suffix_len = len(CHAT_SUFFIX_IDS)

    c_full_ids = (
        CHAT_PREFIX_IDS
        + c_user
        + CHAT_SUFFIX_IDS
    )

    f_full_ids = (
        CHAT_PREFIX_IDS
        + f_user
        + CHAT_SUFFIX_IDS
    )

    if c_full_ids != shell['official_ids']:
        raise RuntimeError(
            'Canonical full IDs disagree with official Gemma chat.'
        )

    # ---------------------------
    # Source geometries: CC / FC
    # ---------------------------
    cc_pos = np.arange(
        len(c_full_ids),
        dtype=np.float32,
    )

    prefix_pos = np.arange(
        prefix_len,
        dtype=np.float32,
    )

    canonical_suffix_start = (
        prefix_len + len(c_user)
    )

    canonical_suffix_pos = np.arange(
        canonical_suffix_start,
        canonical_suffix_start + suffix_len,
        dtype=np.float32,
    )

    fc_user_pos = (
        prefix_len
        + enc['fragmented_source_positions']
    )

    fc_pos = np.concatenate(
        [
            prefix_pos,
            fc_user_pos,
            canonical_suffix_pos,
        ]
    ).astype(np.float32)

    # ---------------------------
    # Fully expanded geometry: FE
    # ---------------------------
    fe_pos = np.arange(
        len(f_full_ids),
        dtype=np.float32,
    )

    c_filler_idx = np.where(
        c_labels == 'filler'
    )[0]
    c_query_idx = np.where(
        c_labels == 'query'
    )[0]
    f_filler_idx = np.where(
        f_labels == 'filler'
    )[0]
    f_query_idx = np.where(
        f_labels == 'query'
    )[0]

    if not (
        len(c_filler_idx)
        and len(c_query_idx)
        and len(f_filler_idx)
        and len(f_query_idx)
    ):
        raise RuntimeError(
            'Missing canonical/fragmented semantic regions.'
        )

    c_query_first = int(c_query_idx[0])
    f_query_first = int(f_query_idx[0])

    query_offset = (
        f_query_first
        - c_query_first
    )

    if query_offset <= 0:
        raise RuntimeError(
            'Fragmentation did not create positive query displacement.'
        )

    expanded_suffix_start = (
        prefix_len + len(f_user)
    )

    expanded_suffix_pos = np.arange(
        expanded_suffix_start,
        expanded_suffix_start + suffix_len,
        dtype=np.float32,
    )

    # ---------------------------------------------------------
    # CQ: canonical IDs; source evidence/filler, shifted query.
    # ---------------------------------------------------------
    cq_user_pos = np.arange(
        len(c_user),
        dtype=np.float32,
    )

    cq_user_pos[c_query_idx] = (
        c_query_idx.astype(np.float32)
        + query_offset
    )

    cq_pos = np.concatenate(
        [
            prefix_pos,
            prefix_len + cq_user_pos,
            expanded_suffix_pos,
        ]
    ).astype(np.float32)

    # ---------------------------------------------------------
    # CE: same query/readout as CQ, but spread canonical filler.
    # ---------------------------------------------------------
    ce_user_pos = cq_user_pos.copy()

    c_filler_first = int(
        c_filler_idx[0]
    )
    c_filler_n = len(
        c_filler_idx
    )

    ce_user_pos[c_filler_idx] = (
        c_filler_first
        + np.arange(
            c_filler_n,
            dtype=np.float32,
        )
        * (
            f_query_first
            - c_filler_first
        )
        / c_filler_n
    )

    ce_pos = np.concatenate(
        [
            prefix_pos,
            prefix_len + ce_user_pos,
            expanded_suffix_pos,
        ]
    ).astype(np.float32)

    # ---------------------------------------------------------
    # FQ: fragmented IDs; source-aligned evidence/filler,
    #     shifted query/readout.
    # ---------------------------------------------------------
    fq_user_pos = enc[
        'fragmented_source_positions'
    ].copy()

    fq_user_pos[f_query_idx] = (
        fq_user_pos[f_query_idx]
        + query_offset
    )

    fq_pos = np.concatenate(
        [
            prefix_pos,
            prefix_len + fq_user_pos,
            expanded_suffix_pos,
        ]
    ).astype(np.float32)

    maps = {
        'CC': {
            'condition': 'canonical_ids_source_pos',
            'ids': c_full_ids,
            'positions': cc_pos,
            'labels': c_labels,
        },
        'CQ': {
            'condition': 'canonical_ids_query_shift_pos',
            'ids': c_full_ids,
            'positions': cq_pos,
            'labels': c_labels,
        },
        'CE': {
            'condition': 'canonical_ids_filler_spread_pos',
            'ids': c_full_ids,
            'positions': ce_pos,
            'labels': c_labels,
        },
        'FC': {
            'condition': 'fragmented_ids_source_pos',
            'ids': f_full_ids,
            'positions': fc_pos,
            'labels': f_labels,
        },
        'FQ': {
            'condition': 'fragmented_ids_query_shift_pos',
            'ids': f_full_ids,
            'positions': fq_pos,
            'labels': f_labels,
        },
        'FE': {
            'condition': 'fragmented_ids_filler_spread_pos',
            'ids': f_full_ids,
            'positions': fe_pos,
            'labels': f_labels,
        },
    }

    for key, item in maps.items():
        p = item['positions']

        if len(item['ids']) != len(p):
            raise RuntimeError(
                f'{key}: IDs/positions length mismatch.'
            )

        if (
            len(p) > 1
            and not np.all(np.diff(p) > 0)
        ):
            bad = np.where(
                np.diff(p) <= 0
            )[0][:10]

            raise RuntimeError(
                f'{key}: positions not strictly increasing: '
                f'{bad.tolist()}'
            )

        item['ids_hash'] = hash_ids(
            item['ids']
        )
        item['position_hash'] = hash_positions(
            p
        )
        item['gaps'] = full_gap_diagnostics(
            p,
            prefix_len,
            item['labels'],
        )
        item['realized_tokens'] = len(
            item['ids']
        )
        item['position_max'] = float(
            p[-1]
        )

    # Same-token-ID position contrasts.
    if not (
        maps['CC']['ids_hash']
        == maps['CQ']['ids_hash']
        == maps['CE']['ids_hash']
    ):
        raise RuntimeError(
            'CC/CQ/CE must have identical canonical IDs.'
        )

    if not (
        maps['FC']['ids_hash']
        == maps['FQ']['ids_hash']
        == maps['FE']['ids_hash']
    ):
        raise RuntimeError(
            'FC/FQ/FE must have identical fragmented IDs.'
        )

    # Source geometry matched across token identities.
    if abs(
        maps['CC']['gaps']['evidence_readout']
        - maps['FC']['gaps']['evidence_readout']
    ) > 1e-3:
        raise RuntimeError(
            'CC/FC source evidence→readout geometry differs.'
        )

    # Query-shift geometry matched across token identities.
    if abs(
        maps['CQ']['gaps']['evidence_readout']
        - maps['FQ']['gaps']['evidence_readout']
    ) > 1e-3:
        raise RuntimeError(
            'CQ/FQ evidence→readout geometry differs.'
        )

    # Query-only interventions must preserve source filler span.
    if abs(
        maps['CQ']['gaps']['filler_span']
        - maps['CC']['gaps']['filler_span']
    ) > 1e-3:
        raise RuntimeError(
            'CQ unexpectedly changed canonical filler span.'
        )

    if abs(
        maps['FQ']['gaps']['filler_span']
        - maps['FC']['gaps']['filler_span']
    ) > 1e-3:
        raise RuntimeError(
            'FQ unexpectedly changed fragmented source filler span.'
        )

    # CQ/CE and FQ/FE share the same query/readout positions;
    # only filler positions differ.
    for a, b in [
        ('CQ', 'CE'),
        ('FQ', 'FE'),
    ]:
        if abs(
            maps[a]['gaps']['evidence_readout']
            - maps[b]['gaps']['evidence_readout']
        ) > 1e-3:
            raise RuntimeError(
                f'{a}/{b} global geometry differs.'
            )

        if abs(
            maps[a]['position_max']
            - maps[b]['position_max']
        ) > 1e-3:
            raise RuntimeError(
                f'{a}/{b} readout absolute position differs.'
            )

        if not (
            maps[b]['gaps']['filler_span']
            > maps[a]['gaps']['filler_span']
        ):
            raise RuntimeError(
                f'{a}->{b} did not spread filler positions.'
            )

    # Expanded geometry matched across token identities.
    if abs(
        maps['CE']['gaps']['evidence_readout']
        - maps['FE']['gaps']['evidence_readout']
    ) > 1e-3:
        raise RuntimeError(
            'CE/FE expanded evidence→readout geometry differs.'
        )

    if not (
        maps['CQ']['gaps']['evidence_readout']
        > maps['CC']['gaps']['evidence_readout']
    ):
        raise RuntimeError(
            'Query shift did not increase evidence→readout distance.'
        )

    return maps

print('GEMMA SIX-CONDITION GEOMETRY ASSEMBLY READY.')

## 10. Six-condition structural/finite engineering gate at ~1K

In [ ]:
gate_inst = build_instance(
    filler_language='en',
    semantic_seed=6999,
    trial_index=7,
    target_chat_tokens=1024,
)

gate_maps = assemble_geometry(
    gate_inst
)

gate_scores = {}

for key in [
    'CC',
    'CQ',
    'CE',
    'FC',
    'FQ',
    'FE',
]:
    item = gate_maps[key]

    print(
        '\n',
        key,
        {
            'condition': item[
                'condition'
            ],
            'tokens': item[
                'realized_tokens'
            ],
            'position_max': item[
                'position_max'
            ],
            'evidence_readout_gap': item[
                'gaps'
            ][
                'evidence_readout'
            ],
            'filler_span': item[
                'gaps'
            ][
                'filler_span'
            ],
        },
    )

    scored = score_one_shot_custom(
        item['ids'],
        item['positions'],
        gate_inst.gold_digit,
    )

    gate_scores[key] = scored

    print(
        {
            'gold': gate_inst.gold_digit,
            'pred': scored[
                'predicted_digit'
            ],
            'margin': scored[
                'gold_logit_margin'
            ],
            'seconds': scored[
                'seconds'
            ],
            'peak_cuda_gb': scored[
                'peak_cuda_gb'
            ],
        },
    )

    if not (
        np.isfinite(
            scored[
                'gold_logit_margin'
            ]
        )
        and np.isfinite(
            scored[
                'forced_choice_nll'
            ]
        )
    ):
        raise RuntimeError(
            f'Non-finite score in {key}.'
        )

print(
    '\nSIX-CONDITION STRUCTURAL / '
    'FINITE GATE PASSED.'
)

# Phase A — prospective six-condition run

32 semantic trials × 2 filler languages × 2 canonical target lengths × 6 conditions
= **768 rows**.

Rows are appended immediately to Google Drive, so the cell is resumable after interruption.

## 11. Append-only result helpers

In [ ]:
def load_jsonl(path):
    rows = []

    if path.exists():
        with path.open(
            'r',
            encoding='utf-8',
        ) as f:
            for line in f:
                line = line.strip()

                if line:
                    rows.append(
                        json.loads(line)
                    )

    return rows

def completed_ids(path):
    return {
        row['result_id']
        for row in load_jsonl(path)
        if 'result_id' in row
    }

def make_result_id(meta):
    blob = json.dumps(
        meta,
        sort_keys=True,
        separators=(',', ':'),
    ).encode('utf-8')

    return hashlib.sha256(
        blob
    ).hexdigest()[:24]

def append_jsonl(path, row):
    with path.open(
        'a',
        encoding='utf-8',
    ) as f:
        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
            )
            + '\n'
        )
        f.flush()

print(
    'Existing completed rows:',
    len(
        completed_ids(
            RESULTS_PATH
        )
    ),
)

## 12. Main 768-row localization run

In [ ]:
CONDITIONS = [
    'CC',
    'CQ',
    'CE',
    'FC',
    'FQ',
    'FE',
]

done = completed_ids(
    RESULTS_PATH
)

for filler_language in FILLER_LANGUAGES:
    for target_chat_tokens in TARGET_LENGTHS:
        for trial_index, semantic_seed in enumerate(
            SEMANTIC_SEEDS
        ):
            inst = build_instance(
                filler_language=filler_language,
                semantic_seed=semantic_seed,
                trial_index=trial_index,
                target_chat_tokens=target_chat_tokens,
            )

            maps = assemble_geometry(
                inst
            )

            order = CONDITIONS[:]

            stable_rng(
                'condition_order_gemma_v072',
                filler_language,
                target_chat_tokens,
                semantic_seed,
                trial_index,
            ).shuffle(order)

            evidence_text = '\n'.join(
                inst.evidence_lines
            )
            query_text = '\n'.join(
                inst.query_lines
            )

            for order_index, key in enumerate(
                order
            ):
                item = maps[key]

                meta = {
                    'phase': (
                        'gemma_geometry_localization_4k8k'
                    ),
                    'model': MODEL_NAME,
                    'notebook_version': NOTEBOOK_VERSION,
                    'task_version': TASK_VERSION,
                    'scorer_version': SCORER_VERSION,
                    'position_intervention': POSITION_INTERVENTION,
                    'filler_language': filler_language,
                    'target_chat_tokens': target_chat_tokens,
                    'semantic_seed': semantic_seed,
                    'trial_index': trial_index,
                    'n_records': N_RECORDS,
                    'geometry_code': key,
                    'condition': item[
                        'condition'
                    ],
                    'frag_keep': (
                        None
                        if key in {
                            'CC',
                            'CQ',
                            'CE',
                        }
                        else FRAG_KEEP_BY_LANGUAGE[
                            filler_language
                        ]
                    ),
                }

                rid = make_result_id(
                    meta
                )

                if rid in done:
                    continue

                scored = score_one_shot_custom(
                    item['ids'],
                    item['positions'],
                    inst.gold_digit,
                )

                row = {
                    'result_id': rid,
                    **meta,
                    'condition_order_index': order_index,
                    'gold_digit': inst.gold_digit,
                    'target_key': inst.target_key,
                    'position_rank': position_rank_for_trial(
                        trial_index
                    ),
                    'target_record_position': inst.target_record_position,
                    'realized_full_tokens': item[
                        'realized_tokens'
                    ],
                    'position_max': item[
                        'position_max'
                    ],
                    'evidence_query_position_gap': item[
                        'gaps'
                    ][
                        'evidence_query'
                    ],
                    'evidence_readout_position_gap': item[
                        'gaps'
                    ][
                        'evidence_readout'
                    ],
                    'filler_position_span': item[
                        'gaps'
                    ][
                        'filler_span'
                    ],
                    'filler_query_position_gap': item[
                        'gaps'
                    ][
                        'filler_query_gap'
                    ],
                    'ids_hash': item[
                        'ids_hash'
                    ],
                    'position_hash': item[
                        'position_hash'
                    ],
                    'prompt_sha256': hashlib.sha256(
                        inst.user_content.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'evidence_sha256': hashlib.sha256(
                        evidence_text.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'query_sha256': hashlib.sha256(
                        query_text.encode(
                            'utf-8'
                        )
                    ).hexdigest(),
                    'timestamp_utc': (
                        pd.Timestamp.utcnow().isoformat()
                    ),
                    **flatten_score(
                        scored
                    ),
                }

                append_jsonl(
                    RESULTS_PATH,
                    row,
                )
                done.add(
                    rid
                )

                print(
                    f"fill={filler_language} "
                    f"L={target_chat_tokens:5d} "
                    f"trial={trial_index:02d} "
                    f"{key} "
                    f"gold={inst.gold_digit} "
                    f"pred={scored['predicted_digit']} "
                    f"margin={scored['gold_logit_margin']:+8.3f} "
                    f"n={item['realized_tokens']:5d} "
                    f"ERgap={item['gaps']['evidence_readout']:8.1f} "
                    f"Fspan={item['gaps']['filler_span']:8.1f} "
                    f"sec={scored['seconds']:5.2f} "
                    f"peak={scored['peak_cuda_gb']:.2f}GB"
                )

print(
    'Localization run complete:',
    RESULTS_PATH,
)
print(
    'Rows on disk:',
    len(
        load_jsonl(
            RESULTS_PATH
        )
    ),
)

# Phase B — invariants and decomposition

## 13. Pairing, same-ID, geometry, and task-invariance gates

In [ ]:
df = pd.DataFrame(
    load_jsonl(
        RESULTS_PATH
    )
)

expected_rows = (
    len(FILLER_LANGUAGES)
    * len(TARGET_LENGTHS)
    * len(SEMANTIC_SEEDS)
    * 6
)

if len(df) != expected_rows:
    raise RuntimeError(
        f'Expected {expected_rows} rows; '
        f'found {len(df)}.'
    )

if df[
    'result_id'
].duplicated().any():
    raise RuntimeError(
        'Duplicate result IDs found.'
    )

base_cols = [
    'filler_language',
    'target_chat_tokens',
    'semantic_seed',
    'trial_index',
]

condition_counts = df.groupby(
    base_cols
)['geometry_code'].nunique()

if (
    condition_counts != 6
).any():
    raise RuntimeError(
        'Some base instances lack one or more '
        'of the six geometry conditions.'
    )

if (
    df.groupby(
        base_cols
    )[
        'prompt_sha256'
    ].nunique()
    != 1
).any():
    raise RuntimeError(
        'Prompt text differs across geometry conditions.'
    )

pair = df.pivot_table(
    index=base_cols,
    columns='geometry_code',
    values=[
        'ids_hash',
        'evidence_readout_position_gap',
        'filler_position_span',
        'position_max',
    ],
    aggfunc='first',
)

if not (
    (
        pair[
            'ids_hash'
        ]['CC']
        == pair[
            'ids_hash'
        ]['CQ']
    )
    & (
        pair[
            'ids_hash'
        ]['CQ']
        == pair[
            'ids_hash'
        ]['CE']
    )
).all():
    raise RuntimeError(
        'CC/CQ/CE token IDs differ.'
    )

if not (
    (
        pair[
            'ids_hash'
        ]['FC']
        == pair[
            'ids_hash'
        ]['FQ']
    )
    & (
        pair[
            'ids_hash'
        ]['FQ']
        == pair[
            'ids_hash'
        ]['FE']
    )
).all():
    raise RuntimeError(
        'FC/FQ/FE token IDs differ.'
    )

if (
    np.abs(
        pair[
            'evidence_readout_position_gap'
        ]['CC']
        - pair[
            'evidence_readout_position_gap'
        ]['FC']
    ).max()
    > 1e-3
):
    raise RuntimeError(
        'CC/FC source geometry differs.'
    )

if (
    np.abs(
        pair[
            'evidence_readout_position_gap'
        ]['CQ']
        - pair[
            'evidence_readout_position_gap'
        ]['FQ']
    ).max()
    > 1e-3
):
    raise RuntimeError(
        'CQ/FQ query-shift geometry differs.'
    )

if (
    np.abs(
        pair[
            'evidence_readout_position_gap'
        ]['CE']
        - pair[
            'evidence_readout_position_gap'
        ]['FE']
    ).max()
    > 1e-3
):
    raise RuntimeError(
        'CE/FE expanded geometry differs.'
    )

for q_code, e_code in [
    ('CQ', 'CE'),
    ('FQ', 'FE'),
]:
    if (
        np.abs(
            pair[
                'evidence_readout_position_gap'
            ][q_code]
            - pair[
                'evidence_readout_position_gap'
            ][e_code]
        ).max()
        > 1e-3
    ):
        raise RuntimeError(
            f'{q_code}/{e_code} global gap differs.'
        )

    if (
        np.abs(
            pair[
                'position_max'
            ][q_code]
            - pair[
                'position_max'
            ][e_code]
        ).max()
        > 1e-3
    ):
        raise RuntimeError(
            f'{q_code}/{e_code} readout position differs.'
        )

    if not (
        pair[
            'filler_position_span'
        ][e_code]
        > pair[
            'filler_position_span'
        ][q_code]
    ).all():
        raise RuntimeError(
            f'{q_code}->{e_code} did not spread filler.'
        )

task_pair_cols = [
    'target_chat_tokens',
    'semantic_seed',
    'trial_index',
]

if (
    df.groupby(
        task_pair_cols
    )[
        'evidence_sha256'
    ].nunique()
    != 1
).any():
    raise RuntimeError(
        'Task evidence differs across filler languages.'
    )

if (
    df.groupby(
        task_pair_cols
    )[
        'query_sha256'
    ].nunique()
    != 1
).any():
    raise RuntimeError(
        'Task query differs across filler languages.'
    )

if not np.isfinite(
    df[
        [
            'gold_logit_margin',
            'forced_choice_nll',
            'gold_candidate_probability',
        ]
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        'Non-finite outcome metrics found.'
    )

print(
    'PAIRING / SAME-ID / GEOMETRY / '
    'TASK-INVARIANCE GATES PASSED.'
)

## 14. Paired localization effects and exact decomposition

In [ ]:
wide = df.pivot_table(
    index=base_cols,
    columns='geometry_code',
    values=[
        'gold_logit_margin',
        'forced_choice_nll',
        'correct',
    ],
    aggfunc='first',
).reset_index()

for code_name in [
    'CC',
    'CQ',
    'CE',
    'FC',
    'FQ',
    'FE',
]:
    if (
        'gold_logit_margin',
        code_name,
    ) not in wide.columns:
        raise RuntimeError(
            f'Missing {code_name} margin.'
        )

effects = pd.DataFrame({
    'filler_language': wide[
        ('filler_language', '')
    ],
    'target_chat_tokens': wide[
        ('target_chat_tokens', '')
    ],
    'semantic_seed': wide[
        ('semantic_seed', '')
    ],
    'trial_index': wide[
        ('trial_index', '')
    ],
})

CC = wide[
    ('gold_logit_margin', 'CC')
].to_numpy(float)
CQ = wide[
    ('gold_logit_margin', 'CQ')
].to_numpy(float)
CE = wide[
    ('gold_logit_margin', 'CE')
].to_numpy(float)
FC = wide[
    ('gold_logit_margin', 'FC')
].to_numpy(float)
FQ = wide[
    ('gold_logit_margin', 'FQ')
].to_numpy(float)
FE = wide[
    ('gold_logit_margin', 'FE')
].to_numpy(float)

effects[
    'query_shift_canonical'
] = CQ - CC

effects[
    'query_shift_fragmented'
] = FQ - FC

effects[
    'filler_spread_canonical'
] = CE - CQ

effects[
    'filler_spread_fragmented'
] = FE - FQ

effects[
    'total_expansion_canonical'
] = CE - CC

effects[
    'total_expansion_fragmented'
] = FE - FC

effects[
    'representation_source'
] = FC - CC

effects[
    'representation_query_shift'
] = FQ - CQ

effects[
    'representation_spread'
] = FE - CE

effects[
    'query_shift_interaction'
] = (
    (FQ - FC)
    - (CQ - CC)
)

effects[
    'filler_spread_interaction'
] = (
    (FE - FQ)
    - (CE - CQ)
)

for code_name, arr in [
    ('CC', CC),
    ('CQ', CQ),
    ('CE', CE),
    ('FC', FC),
    ('FQ', FQ),
    ('FE', FE),
]:
    effects[
        f'{code_name}_margin'
    ] = arr

canonical_decomp_error = np.max(
    np.abs(
        effects[
            'total_expansion_canonical'
        ].to_numpy()
        - (
            effects[
                'query_shift_canonical'
            ].to_numpy()
            + effects[
                'filler_spread_canonical'
            ].to_numpy()
        )
    )
)

fragmented_decomp_error = np.max(
    np.abs(
        effects[
            'total_expansion_fragmented'
        ].to_numpy()
        - (
            effects[
                'query_shift_fragmented'
            ].to_numpy()
            + effects[
                'filler_spread_fragmented'
            ].to_numpy()
        )
    )
)

if canonical_decomp_error > 1e-9:
    raise RuntimeError(
        'Canonical decomposition identity failed.'
    )

if fragmented_decomp_error > 1e-9:
    raise RuntimeError(
        'Fragmented decomposition identity failed.'
    )

print(
    'Exact canonical decomposition max error =',
    canonical_decomp_error,
)
print(
    'Exact fragmented decomposition max error =',
    fragmented_decomp_error,
)

display(
    effects.head(20)
)

## 15. Cluster bootstrap and language×length cells

In [ ]:
EFFECT_NAMES = [
    'query_shift_canonical',
    'query_shift_fragmented',
    'filler_spread_canonical',
    'filler_spread_fragmented',
    'total_expansion_canonical',
    'total_expansion_fragmented',
    'representation_source',
    'representation_query_shift',
    'representation_spread',
    'query_shift_interaction',
    'filler_spread_interaction',
]

def bootstrap_mean_ci(
    values,
    n_boot=BOOTSTRAP_REPS,
    seed=BOOTSTRAP_SEED,
):
    x = np.asarray(
        pd.Series(
            values
        ).dropna(),
        dtype=float,
    )

    rng = np.random.default_rng(
        seed
    )

    idx = rng.integers(
        0,
        len(x),
        size=(
            n_boot,
            len(x),
        ),
    )

    boot = x[
        idx
    ].mean(
        axis=1
    )

    return (
        float(
            x.mean()
        ),
        float(
            np.quantile(
                boot,
                0.025,
            )
        ),
        float(
            np.quantile(
                boot,
                0.975,
            )
        ),
    )

def cluster_bootstrap_mean_ci(
    frame,
    value_col,
    cluster_col='trial_index',
    n_boot=BOOTSTRAP_REPS,
    seed=BOOTSTRAP_SEED,
):
    clean = frame[
        [
            cluster_col,
            value_col,
        ]
    ].dropna()

    clusters = np.asarray(
        sorted(
            clean[
                cluster_col
            ].unique()
        )
    )

    grouped = {
        c: clean.loc[
            clean[
                cluster_col
            ] == c,
            value_col,
        ].to_numpy(
            dtype=float
        )
        for c in clusters
    }

    observed = float(
        clean[
            value_col
        ].mean()
    )

    rng = np.random.default_rng(
        seed
    )

    boot = np.empty(
        n_boot
    )

    for b in range(
        n_boot
    ):
        sampled = rng.choice(
            clusters,
            size=len(
                clusters
            ),
            replace=True,
        )

        vals = np.concatenate(
            [
                grouped[c]
                for c in sampled
            ]
        )

        boot[b] = vals.mean()

    return (
        observed,
        float(
            np.quantile(
                boot,
                0.025,
            )
        ),
        float(
            np.quantile(
                boot,
                0.975,
            )
        ),
    )

cell_rows = []

for (
    filler_language,
    target_chat_tokens,
), g in effects.groupby(
    [
        'filler_language',
        'target_chat_tokens',
    ]
):
    for effect in EFFECT_NAMES:
        mean, lo, hi = bootstrap_mean_ci(
            g[
                effect
            ],
            seed=(
                BOOTSTRAP_SEED
                + stable_int(
                    filler_language,
                    target_chat_tokens,
                    effect,
                )
                % 100000
            ),
        )

        cell_rows.append({
            'filler_language': filler_language,
            'target_chat_tokens': int(
                target_chat_tokens
            ),
            'effect': effect,
            'n': len(g),
            'mean': mean,
            'ci_lo': lo,
            'ci_hi': hi,
            'positive_n': int(
                (
                    g[
                        effect
                    ] > 0
                ).sum()
            ),
            'negative_n': int(
                (
                    g[
                        effect
                    ] < 0
                ).sum()
            ),
        })

cell_summary = pd.DataFrame(
    cell_rows
)

display(
    cell_summary
)

pooled_rows = []

for effect in EFFECT_NAMES:
    mean, lo, hi = cluster_bootstrap_mean_ci(
        effects,
        effect,
        seed=(
            BOOTSTRAP_SEED
            + stable_int(
                'pooled',
                effect,
            )
            % 100000
        ),
    )

    pooled_rows.append({
        'effect': effect,
        'n_rows': len(
            effects
        ),
        'n_trial_clusters': effects[
            'trial_index'
        ].nunique(),
        'mean': mean,
        'ci_lo': lo,
        'ci_hi': hi,
        'positive_n': int(
            (
                effects[
                    effect
                ] > 0
            ).sum()
        ),
        'negative_n': int(
            (
                effects[
                    effect
                ] < 0
            ).sum()
        ),
    })

pooled_summary = pd.DataFrame(
    pooled_rows
)

display(
    pooled_summary
)

for _, row in pooled_summary.iterrows():
    print(
        f"{row['effect']:<42} "
        f"mean={row['mean']:+.4f} "
        f"95% cluster-CI=["
        f"{row['ci_lo']:+.4f},"
        f"{row['ci_hi']:+.4f}] "
        f"positive={int(row['positive_n'])}/"
        f"{int(row['n_rows'])}"
    )

## 16. Leave-one-trial-out sign stability

In [ ]:
loo_rows = []

for left_out in sorted(
    effects[
        'trial_index'
    ].unique()
):
    g = effects[
        effects[
            'trial_index'
        ] != left_out
    ]

    row = {
        'left_out_trial': int(
            left_out
        ),
    }

    for effect in EFFECT_NAMES:
        row[
            effect
        ] = float(
            g[
                effect
            ].mean()
        )

    loo_rows.append(
        row
    )

loo = pd.DataFrame(
    loo_rows
)

loo_summary_rows = []

for effect in EFFECT_NAMES:
    loo_summary_rows.append({
        'effect': effect,
        'full_mean': float(
            effects[
                effect
            ].mean()
        ),
        'loo_min': float(
            loo[
                effect
            ].min()
        ),
        'loo_max': float(
            loo[
                effect
            ].max()
        ),
        'loo_positive': bool(
            (
                loo[
                    effect
                ] > 0
            ).all()
        ),
        'loo_negative': bool(
            (
                loo[
                    effect
                ] < 0
            ).all()
        ),
    })

loo_summary = pd.DataFrame(
    loo_summary_rows
)

display(
    loo_summary
)

## 17. Cell means and sign counts

In [ ]:
cell_wide = cell_summary.pivot_table(
    index=[
        'filler_language',
        'target_chat_tokens',
    ],
    columns='effect',
    values='mean',
    aggfunc='first',
).reset_index()

display(
    cell_wide
)

positive_cells = {}
negative_cells = {}

for effect in EFFECT_NAMES:
    positive_cells[
        effect
    ] = int(
        (
            cell_wide[
                effect
            ] > 0
        ).sum()
    )

    negative_cells[
        effect
    ] = int(
        (
            cell_wide[
                effect
            ] < 0
        ).sum()
    )

print(
    'Positive cells:',
    positive_cells,
)
print(
    'Negative cells:',
    negative_cells,
)

# Phase C — prospective Gemma sign-pattern replication and localization

## 18. Replication gate and robust component directions

In [ ]:
cc_rows = df[
    df[
        'geometry_code'
    ] == 'CC'
]

canonical_acc = float(
    cc_rows[
        'correct'
    ].mean()
)

cc_pred_counts = (
    cc_rows[
        'predicted_digit'
    ]
    .value_counts()
    .reindex(
        DIGITS,
        fill_value=0,
    )
)

canonical_max_pred_share = float(
    cc_pred_counts.max()
    / cc_pred_counts.sum()
)

pooled = pooled_summary.set_index(
    'effect'
)

loo_indexed = loo_summary.set_index(
    'effect'
)

def pooled_mean(effect):
    return float(
        pooled.loc[
            effect,
            'mean',
        ]
    )

def pooled_lo(effect):
    return float(
        pooled.loc[
            effect,
            'ci_lo',
        ]
    )

def pooled_hi(effect):
    return float(
        pooled.loc[
            effect,
            'ci_hi',
        ]
    )

def loo_positive(effect):
    return bool(
        loo_indexed.loc[
            effect,
            'loo_positive',
        ]
    )

def loo_negative(effect):
    return bool(
        loo_indexed.loc[
            effect,
            'loo_negative',
        ]
    )

SIGN_PATTERN_REPLICATION = bool(
    canonical_acc >= 0.90
    and canonical_max_pred_share <= 0.40
    and pooled_mean(
        'total_expansion_canonical'
    ) < 0
    and pooled_mean(
        'total_expansion_fragmented'
    ) > 0
    and negative_cells[
        'total_expansion_canonical'
    ] >= 3
    and positive_cells[
        'total_expansion_fragmented'
    ] >= 3
    and loo_negative(
        'total_expansion_canonical'
    )
    and loo_positive(
        'total_expansion_fragmented'
    )
)

def robust_direction(effect):
    if (
        pooled_mean(effect) > 0
        and pooled_lo(effect) > 0
        and positive_cells[
            effect
        ] >= 3
        and loo_positive(
            effect
        )
    ):
        return 'positive'

    if (
        pooled_mean(effect) < 0
        and pooled_hi(effect) < 0
        and negative_cells[
            effect
        ] >= 3
        and loo_negative(
            effect
        )
    ):
        return 'negative'

    return 'unresolved'

component_directions = {
    effect: robust_direction(
        effect
    )
    for effect in [
        'query_shift_canonical',
        'filler_spread_canonical',
        'query_shift_fragmented',
        'filler_spread_fragmented',
        'query_shift_interaction',
        'filler_spread_interaction',
        'representation_source',
        'representation_query_shift',
        'representation_spread',
    ]
}

def summarize_path(
    query_direction,
    spread_direction,
    expected_total_direction,
):
    if (
        query_direction
        == expected_total_direction
        and spread_direction
        == expected_total_direction
    ):
        return 'both_components_same_direction'

    if (
        query_direction
        == expected_total_direction
        and spread_direction
        == 'unresolved'
    ):
        return 'query_shift_localized'

    if (
        spread_direction
        == expected_total_direction
        and query_direction
        == 'unresolved'
    ):
        return 'filler_spread_localized'

    if (
        query_direction
        != 'unresolved'
        and spread_direction
        != 'unresolved'
        and query_direction
        != spread_direction
    ):
        return 'opposing_components'

    return 'mixed_or_unresolved'

canonical_localization = summarize_path(
    component_directions[
        'query_shift_canonical'
    ],
    component_directions[
        'filler_spread_canonical'
    ],
    'negative',
)

fragmented_localization = summarize_path(
    component_directions[
        'query_shift_fragmented'
    ],
    component_directions[
        'filler_spread_fragmented'
    ],
    'positive',
)

print(
    'CC accuracy:',
    canonical_acc,
)
print(
    'CC prediction counts:',
    cc_pred_counts.to_dict(),
)
print(
    'CC max prediction share:',
    canonical_max_pred_share,
)

print(
    '\nTotal effects:'
)
for effect in [
    'total_expansion_canonical',
    'total_expansion_fragmented',
]:
    print(
        f"{effect:<34} "
        f"mean={pooled_mean(effect):+.4f} "
        f"CI=[{pooled_lo(effect):+.4f},"
        f"{pooled_hi(effect):+.4f}] "
        f"+cells={positive_cells[effect]} "
        f"-cells={negative_cells[effect]} "
        f"LOO+={loo_positive(effect)} "
        f"LOO-={loo_negative(effect)}"
    )

print(
    '\nComponent directions:',
    component_directions,
)

print(
    'Canonical localization:',
    canonical_localization,
)
print(
    'Fragmented localization:',
    fragmented_localization,
)

print(
    '\nSIGN_PATTERN_REPLICATION =',
    SIGN_PATTERN_REPLICATION,
)

if SIGN_PATTERN_REPLICATION:
    print(
        '\nGemma v0.7.1 sign interaction '
        'replicated on fresh seeds. '
        'Component localization is eligible '
        'for confirmatory interpretation.'
    )
else:
    print(
        '\nGemma v0.7.1 sign interaction did '
        'not satisfy the fresh-seed gate. '
        'Treat component decomposition as '
        'descriptive and do not retune.'
    )

## 19. Compare fresh-seed totals with v0.7.1

In [ ]:
comparison = {
    'v071': {
        'canonical_total': {
            'mean': float(
                prior_can[
                    'mean'
                ]
            ),
            'ci_lo': float(
                prior_can[
                    'ci_lo'
                ]
            ),
            'ci_hi': float(
                prior_can[
                    'ci_hi'
                ]
            ),
        },
        'fragmented_total': {
            'mean': float(
                prior_frag[
                    'mean'
                ]
            ),
            'ci_lo': float(
                prior_frag[
                    'ci_lo'
                ]
            ),
            'ci_hi': float(
                prior_frag[
                    'ci_hi'
                ]
            ),
        },
    },
    'v072': {
        'canonical_total': {
            'mean': pooled_mean(
                'total_expansion_canonical'
            ),
            'ci_lo': pooled_lo(
                'total_expansion_canonical'
            ),
            'ci_hi': pooled_hi(
                'total_expansion_canonical'
            ),
        },
        'fragmented_total': {
            'mean': pooled_mean(
                'total_expansion_fragmented'
            ),
            'ci_lo': pooled_lo(
                'total_expansion_fragmented'
            ),
            'ci_hi': pooled_hi(
                'total_expansion_fragmented'
            ),
        },
    },
}

print(
    json.dumps(
        comparison,
        indent=2,
    )
)

## 20. Save v0.7.2 audit

In [ ]:
pooled_effects = {}

for _, row in pooled_summary.iterrows():
    pooled_effects[
        row[
            'effect'
        ]
    ] = {
        'mean': float(
            row[
                'mean'
            ]
        ),
        'ci_lo': float(
            row[
                'ci_lo'
            ]
        ),
        'ci_hi': float(
            row[
                'ci_hi'
            ]
        ),
    }

audit = {
    'model': MODEL_NAME,
    'transformers_version': transformers.__version__,
    'torch_version': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'notebook_version': NOTEBOOK_VERSION,
    'task_version': TASK_VERSION,
    'scorer_version': SCORER_VERSION,
    'confirmatory_scorer': (
        'one_shot_no_cache_custom_masks'
    ),
    'precision_mode': engineering[
        'precision_mode'
    ],
    'position_intervention': POSITION_INTERVENTION,
    'followup_status': (
        'prospective_fresh_seed_gemma_geometry_localization'
    ),
    'previous_v071_cross_model_position_replication': previous[
        'cross_model_position_replication'
    ],
    'previous_v071_total_effects': {
        'canonical': prior_can,
        'fragmented': prior_frag,
    },
    'claim_scope': 'associative_retrieval',
    'frozen_n_records': N_RECORDS,
    'frozen_frag_keep_by_language': FRAG_KEEP_BY_LANGUAGE,
    'new_semantic_seeds': SEMANTIC_SEEDS,
    'target_lengths': TARGET_LENGTHS,
    'filler_languages': FILLER_LANGUAGES,
    'primary_metric': (
        'gold_vs_best_distractor_logit_margin'
    ),
    'expected_rows': int(
        expected_rows
    ),
    'observed_rows': int(
        len(df)
    ),
    'gold_position_combinations_each_n': 2,
    'same_ids_CC_CQ_CE': True,
    'same_ids_FC_FQ_FE': True,
    'CC_FC_same_source_geometry': True,
    'CQ_FQ_same_query_shift_global_geometry': True,
    'CE_FE_same_expanded_global_geometry': True,
    'CQ_CE_same_query_readout_geometry': True,
    'FQ_FE_same_query_readout_geometry': True,
    'exact_decomposition_canonical': True,
    'exact_decomposition_fragmented': True,
    'canonical_accuracy': canonical_acc,
    'canonical_max_pred_share': canonical_max_pred_share,
    'pooled_effects_cluster_bootstrap': pooled_effects,
    'positive_cells': positive_cells,
    'negative_cells': negative_cells,
    'leave_one_trial_out': {
        row[
            'effect'
        ]: {
            'loo_min': float(
                row[
                    'loo_min'
                ]
            ),
            'loo_max': float(
                row[
                    'loo_max'
                ]
            ),
            'loo_positive': bool(
                row[
                    'loo_positive'
                ]
            ),
            'loo_negative': bool(
                row[
                    'loo_negative'
                ]
            ),
        }
        for _, row in loo_summary.iterrows()
    },
    'sign_pattern_replication': bool(
        SIGN_PATTERN_REPLICATION
    ),
    'component_robust_directions': component_directions,
    'canonical_localization': canonical_localization,
    'fragmented_localization': fragmented_localization,
}

with AUDIT_PATH.open(
    'w',
    encoding='utf-8',
) as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False,
    )
)

print(
    'Saved:',
    AUDIT_PATH,
)

# Files to share back

After the notebook completes, upload:

1. `v072_gemma_geometry_localization_4k8k.jsonl`
2. `v072_gemma_geometry_localization_audit.json`

The primary gate is:

`sign_pattern_replication: true/false`

Only if that is `true` should the component localization be treated as a prospective
localization of the Gemma v0.7.1 sign interaction.